# Annotated original: CO2 adsorption notebook
All original source cells are preserved below. Each is followed by a line-numbered explanation and keep/delete/change notes. The original source cell numbers count both Markdown and code. This is a reading copy, not a leakage-safe corrected analysis. Original bulky outputs were cleared to avoid implying this annotated copy was executed.

See the companion review for validation results, exact deletion instructions, and improved-validation alternatives.


<a href="https://colab.research.google.com/github/hpedrazamartinez/ENCH680-CO2-Adsorption-ML/blob/main/ENCH680_CO2_Adsorption_Reproduction_Henry_and_Kyra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Explanation: original cell 1

- **Line 1 — Optional:** HTML hyperlink containing the Colab badge image. href opens this notebook on GitHub in Google Colab; target controls where it opens; img supplies the badge; alt supplies fallback text. This affects navigation, not calculations.


# ENCH 680 Course Project: Reproducing Yuan et al. (2021)
**Applied Machine Learning for Prediction of CO₂ Adsorption on Biomass Waste-Derived Porous Carbons**
*Environ. Sci. Technol.* 2021, 55, 11925–11936. https://doi.org/10.1021/acs.est.1c01849

Group members: Kyra Kjargaard, Henry Pedraza Martinez. University of Calgary, ENCH 680.

**What this notebook does**
1. Loads the authors' dataset (527 data points), [authors' GitHub repository](https://github.com/ssuvarnamanu/CO2-adsorption-by-biowaste-based-porous-carbon).
2. Repeats the preprocessing: random-forest imputation of TPV and MPV from SA, then drops S (%).
3. Trains the Chapter 3 baselines (a single decision tree with and without pre-pruning, and a random forest).
4. Reproduces the paper's three boosted-tree models (GBDT, XGBoost, LightGBM) with an 85/15 split, 5-fold CV and grid search.
5. Compares our results with Table 1 of the paper.
6. Interprets the model with permutation importance (MDA) and partial dependence plots (PDPs).
7. Repeats the analysis on the RPC and HDPC sub-datasets.
8. Tests improvements: split sensitivity, leakage-safe grouped CV, SHAP, and the extrapolation limits of trees.

### Explanation: original cell 2

- **Line 1 — Keep:** Markdown project title; # means a level-one heading. It describes a reproduction study, not executable Python.
- **Line 2 — Keep:** Bold formatting around the paper title. This identifies the study being reproduced.
- **Line 3 — Keep:** Italic journal name followed by bibliographic details and the DOI link. These identify the original paper; they do not load any data.
- **Line 4 — Optional:** Blank line separates Markdown paragraphs. It has no computational effect.
- **Line 5 — Keep:** Project authorship and affiliation. This is attribution, not a Python variable.
- **Line 6 — Optional:** Blank line improves readability.
- **Line 7 — Keep:** Bold label introducing the promised workflow.
- **Line 8 — Change:** Numbered-list item describing the data. There are 527 rows, but the actual download comes from your GitHub copy, not directly from the linked authors’ repository. The downloaded workbook was byte-identical to the authors’ workbook when checked.
- **Line 9 — Keep + caveat:** Describes the implemented RF filling of missing pore volumes and sulfur removal. This is a reproduction choice; learning the imputation before the outer split is not leakage-safe.
- **Line 10 — Keep:** Matches the implemented unpruned tree, tuned pre-pruned tree, and tuned random-forest baselines. The course chapter itself was not supplied for independent verification.
- **Line 11 — Change:** Broadly matches the three model families and the 85/15 holdout, but this notebook uses shuffled CV whereas the authors’ released notebook uses cv=5 without additional shuffling. Say approximate reproduction with modern libraries, not identical methodology.
- **Line 12 — Keep + verify:** The comparison is implemented, but paper values are entered manually in Cell 19. All entered Table 1 values were independently verified against the full publication.
- **Line 13 — Change:** Only permutation importance is implemented. Delete “and partial dependence plots (PDPs)” unless PDP code is added.
- **Line 14 — Delete or implement:** No RPC/HDPC subset analysis exists in the attached notebook. Remove this promise or implement the analysis using verified subset labels.
- **Line 15 — Delete or implement:** Split sensitivity, grouped CV, SHAP, and extrapolation tests are not implemented. Installing SHAP and importing GroupKFold do not perform these analyses.


## 0. Install packages
Required packages: `pip install scikit-learn xgboost lightgbm shap pandas matplotlib seaborn openpyxl`

### Explanation: original cell 3

- **Line 1 — Keep:** Markdown heading introducing environment setup.
- **Line 2 — Change:** A textual package list, not an executed installation. Remove shap and seaborn for the current notebook. The actual install cell also assumes NumPy, pandas, matplotlib, and scikit-learn already exist, which is normally true in Colab but not every new environment.


In [ ]:
%pip install -q xgboost lightgbm shap openpyxl

### Explanation: original cell 4

- **Line 1 — Change:** IPython %pip installs into the active notebook environment; -q reduces installer output. xgboost and lightgbm provide the two boosters; openpyxl reads .xlsx files through pandas. shap is unused and can be removed. This is notebook magic, not ordinary standalone Python syntax. Versions are unpinned.


In [ ]:
import warnings, json, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import (train_test_split, GridSearchCV, cross_val_score, KFold, GroupKFold, cross_validate)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import sklearn, xgboost, lightgbm

SEED = 42
os.makedirs("figs", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__, "| lightgbm", lightgbm.__version__)
RESULTS = {}

### Explanation: original cell 5

- **Line 1 — Change:** Imports warnings for warning control, json for the results export, and os for folder creation. Keep json and os. warnings is unnecessary if the following suppression line is removed.
- **Line 2 — Delete recommended:** Globally hides every Python warning for the rest of the session. It does not fix problems and may conceal compatibility or data warnings. Removing it does not intentionally change model mathematics.
- **Line 3 — Keep:** Imports NumPy as np for arrays, square roots, and plotting positions, and pandas as pd for spreadsheet and table operations. Combining imports on one line is valid, although separate lines are easier to read.
- **Line 4 — Change:** Imports pyplot as plt for every chart, and seaborn as sns. plt is used; sns is never used in the attached notebook and can be removed.
- **Line 5 — Change:** train_test_split creates holdouts; GridSearchCV tunes parameters; cross_val_score calculates fold scores; KFold defines folds. GroupKFold and cross_validate are imported but never called; remove just those two names unless adding the missing analyses.
- **Line 6 — Keep:** Imports coefficient of determination, mean squared error, and mean absolute error. All three are used. RMSE is calculated later by taking the square root of MSE.
- **Line 7 — Keep:** Imports the single regression-tree estimator for the two decision-tree baselines.
- **Line 8 — Keep:** Imports a forest that averages trees and a gradient-boosted sequence of regression trees. Both are used.
- **Line 9 — Keep:** Imports the straight-line regressor used only in the imputation-method comparison. It does not become the final CO2 model.
- **Line 10 — Change:** permutation_importance is used in Cell 23. PartialDependenceDisplay is never used; remove it if PDPs are out of scope, or retain it and add actual PDP code.
- **Line 11 — Keep:** Imports the XGBoost scikit-learn-compatible regression estimator.
- **Line 12 — Keep:** Imports the LightGBM scikit-learn-compatible regression estimator.
- **Line 13 — Keep:** Imports package modules so their __version__ strings can be printed and saved. This is not redundant with importing their estimator classes.
- **Line 14 — Optional:** Blank line separates imports from configuration; it can remain or be removed.
- **Line 15 — Keep:** Sets an integer reused as random_state. Passing 42 explicitly makes relevant pseudo-random operations repeatable within a compatible environment; it is not a guarantee of identical results across software versions or platforms.
- **Line 16 — Keep if saving:** Creates the figs directory if it does not already exist. exist_ok=True avoids an error when rerunning. Keep while any plt.savefig path uses this directory.
- **Line 17 — Optional:** Sets default display resolution and font size, and hides the top and right plot borders. This affects appearance, not fitted predictions. Later savefig(dpi=200) overrides figure.dpi for saved files.
- **Line 18 — Keep recommended:** Prints installed scikit-learn, XGBoost, and LightGBM versions. Useful for diagnosing reproducibility differences; it does not pin or install those versions.
- **Line 19 — Keep if exporting:** Initializes an empty dictionary collecting metrics, parameters, and versions for results.json. Do not remove this alone while later RESULTS assignments remain.


## 1. Data Loading

### Explanation: original cell 6

- **Line 1 — Keep:** Section heading for loading the workbook; no computation occurs.


In [ ]:
import urllib.request
GITHUB_USER = "hpedrazamartinez"
REPO = "ENCH680-CO2-Adsorption-ML"
url = f"https://raw.githubusercontent.com/hpedrazamartinez/ENCH680-CO2-Adsorption-ML/main/data.xlsx"
urllib.request.urlretrieve(url, "data.xlsx")
print("Downloaded data.xlsx from", url)

### Explanation: original cell 7

- **Line 1 — Keep:** Imports Python’s standard-library URL-request module for downloading the workbook. No separate pip package is needed.
- **Line 2 — Delete safe:** Defines a GitHub username string, but the URL below hard-codes the username instead of reading this variable. Nothing later uses GITHUB_USER.
- **Line 3 — Delete safe:** Defines the repository name, but nothing later reads REPO. Either delete it or interpolate both variables into the URL.
- **Line 4 — Change optional:** Stores the download address. The f prefix is unnecessary because there are no {...} replacements. Either use a normal string or use f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data.xlsx". The mutable main branch is not a frozen data version.
- **Line 5 — Keep:** Downloads the URL into a local file named data.xlsx, replacing that file if it already exists. This needs internet and raises an exception on download failure; its returned path/header tuple is intentionally unused.
- **Line 6 — Optional:** Prints the source URL as a download confirmation. Removing this changes only visible logging.


In [ ]:
raw = pd.read_excel("data.xlsx")
print(raw.shape)
raw.describe().T.round(2)

### Explanation: original cell 8

- **Line 1 — Keep:** Reads the first worksheet from the downloaded Excel workbook into DataFrame raw. With this file, raw has 527 rows and 11 columns.
- **Line 2 — Optional:** Prints (rows, columns). This is a useful sanity check, not an input to fitting.
- **Line 3 — Optional recommended:** Computes numeric summary statistics; .T swaps rows and columns so each feature occupies a row; .round(2) rounds the displayed result. As the last cell expression, it is rendered automatically in Jupyter. raw itself is not rounded or modified.


##2. Preprocessing (same as the paper's Supporting Information, Tables S3 and S4)
TPV and MPV are strongly correlated with surface area (SA), so the authors imputed the missing values with a random forest that uses SA as the only input (80/20 split). After that, S (%) is dropped, which leaves 9 input features.

### Explanation: original cell 9

- **Line 1 — Change:** Add a space after ## so this reliably renders as a Markdown heading. “Same as” is too strong for every detail: the released authors’ imputation comparison includes additional models and CV, while yours is simplified.
- **Line 2 — Keep + caveat:** SA means specific surface area; TPV means total pore volume; MPV means micropore volume. The code fills missing TPV/MPV from SA, then removes sulfur and retains nine predictors. This does not prove RF is the best imputer for both properties or that missingness is harmless.


In [ ]:
SA = "Surface Area (m2/g)"; TPV = "Total Pore Volume(cm3/g)"; MPV = "Micropore Volume (cm3/g)"
TARGET = "CO2 uptake (mmol/g)"
df = raw.copy()
imp_rows = []
for col in [TPV, MPV]:
    known = df[[SA, col]].dropna()
    Xk, yk = known[[SA]].values, known[col].values
    Xtr, Xte, ytr, yte = train_test_split(Xk, yk, test_size=0.2, random_state=SEED)
    for name, m in [("Linear regression", LinearRegression()), ("Random forest", RandomForestRegressor(random_state=SEED))]:
        m.fit(Xtr, ytr)
        imp_rows.append([col, name, r2_score(ytr, m.predict(Xtr)), r2_score(yte, m.predict(Xte))])
    rf_imp = RandomForestRegressor(random_state=SEED).fit(Xtr, ytr)
    miss = df[col].isna()
    df.loc[miss, col] = rf_imp.predict(df.loc[miss, [SA]].values)
    print(f"{col}: imputed {miss.sum()} values")
imp = pd.DataFrame(imp_rows, columns=["feature", "model", "train R2", "test R2"]).round(3)
RESULTS["imputation"] = imp.to_dict(orient="records")
df = df.drop(columns=["S (%)"])
FEATURES = [c for c in df.columns if c != TARGET]
print("Features used:", FEATURES)
imp

### Explanation: original cell 10

- **Line 1 — Keep:** Creates three exact column-name aliases on one line, separated by semicolons. SA, TPV, and MPV are strings naming columns, not their numerical contents. Separate assignments would be easier to read.
- **Line 2 — Keep:** Stores the exact name of the response column. CO2 uptake is measured in mmol/g.
- **Line 3 — Keep:** Copies raw into a working DataFrame so filling values and dropping columns do not overwrite the original table. Keeping raw is important for checks and a later leakage-safe analysis.
- **Line 4 — Keep if comparing:** Creates a list that will hold the four imputation-method comparison rows: two target pore-volume columns times two algorithms.
- **Line 5 — Keep:** Starts a loop that first processes total pore volume and then micropore volume. Indentation determines which following statements repeat.
- **Line 6 — Keep:** Selects SA and the current pore-volume column and removes rows missing either. This produces observed training examples for that property. It is not deleting these rows from the main df.
- **Line 7 — Keep:** Double brackets keep SA as a two-dimensional one-column matrix; single brackets make the target one-dimensional. .values converts the pandas objects to NumPy arrays. The paired assignment sets Xk and yk.
- **Line 8 — Keep for reproduction; change for strict validation:** Splits the observed pore-volume examples 80/20 for checking the imputer. These are not the later CO2 train/test sets. Because this happens before the outer split, observed descriptor values from future CO2 test rows can enter imputer fitting.
- **Line 9 — Keep if comparing:** Loops over two named estimator objects, a linear regression and a default 100-tree random forest with a fixed seed. name is a display label; m is the current model.
- **Line 10 — Keep:** Fits the current imputation-comparison model using SA as the predictor and the observed current pore volume as the target. CO2 uptake is not used here.
- **Line 11 — Keep if comparing:** Predicts on the imputer’s training and validation subsets, computes their R2 scores, and appends a row containing feature name, model label, training R2, and validation R2. The training value is an in-sample score, not cross-validated R2.
- **Line 12 — Replace redundant fit:** Fits another RF with the same data and seed as the RF just trained. In this exact two-item loop, m already refers to that fitted RF, so rf_imp = m gives the same imputation without fitting twice. That reuse becomes fragile if the loop order changes.
- **Line 13 — Keep:** Creates a Boolean mask: True where the current pore-volume column is missing, False where measured values already exist.
- **Line 14 — Keep + validate:** Predicts the missing pore-volume entries from their SA values, then writes predictions only into masked rows. Measured pore volumes are preserved. It assumes the required SA values are present and that at least one row is missing.
- **Line 15 — Optional:** Prints which property was processed and the number of missing values filled; summing a Boolean mask counts True entries.
- **Line 16 — Keep if comparing:** Converts the collected comparison rows into a named-column DataFrame and rounds it to three decimals. Here the stored imp table is rounded; the df pore-volume values are not.
- **Line 17 — Keep if exporting:** Converts the comparison table into a list of row dictionaries and stores it in RESULTS. orient="records" means one dictionary per row.
- **Line 18 — Keep for reproduction:** Returns a new DataFrame without the sulfur column and assigns it to df. This mirrors the reproduction’s feature choice, not proof that sulfur has no value; sulfur is not a constant column.
- **Line 19 — Keep:** List comprehension selects every column name except the target. After sulfur removal it produces nine predictors. Excluding TARGET is essential to avoid directly feeding the answer into the model.
- **Line 20 — Optional recommended:** Prints the predictor list as a check that the target and sulfur are absent.
- **Line 21 — Optional:** Displays the imputation comparison table as the last expression in the cell. This is not a fitting step.


##3. Train/test split (85/15, random_state=42, same as the authors' code)

### Explanation: original cell 11

- **Line 1 — Change formatting:** Add a space after ##. The 85/15 split and random_state=42 match the released authors’ split; that does not mean all preprocessing and CV choices match.


In [ ]:
df[FEATURES] = df[FEATURES].astype(float)
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)
cv5 = KFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate(name, model, Xtr=None, ytr=None, Xte=None, yte=None, store=True):
    Xtr = X_train if Xtr is None else Xtr; ytr = y_train if ytr is None else ytr
    Xte = X_test if Xte is None else Xte; yte = y_test if yte is None else yte
    model.fit(Xtr, ytr)
    p_tr, p_te = model.predict(Xtr), model.predict(Xte)
    cv = cross_val_score(model, Xtr, ytr, cv=cv5, scoring="r2")
    row = {"model": name,
           "train R2": r2_score(ytr, p_tr),
           "5-fold CV R2": cv.mean(), "CV R2 std": cv.std(),
           "test R2": r2_score(yte, p_te),
           "test RMSE": np.sqrt(mean_squared_error(yte, p_te)),
           "test MAE": mean_absolute_error(yte, p_te)}
    return row, model, p_tr, p_te

### Explanation: original cell 12

- **Line 1 — Optional for this file:** Converts all predictor columns to floating-point numbers. They are already numeric in the checked workbook, so this is not needed to make these estimators work. It neither fills NaNs nor guarantees finite values.
- **Line 2 — Keep:** Assigns predictor matrix X and response vector y. pandas retains row indexes, so train/test subsets remain traceable to raw rows.
- **Line 3 — Keep:** Randomly partitions aligned X/y rows, reserving 15% for final testing. This dataset yields 447 training and 80 test rows. The default shuffle=True belongs to this holdout split, separate from KFold shuffling.
- **Line 4 — Optional recommended:** Prints the training and test matrix dimensions, confirming nine predictors on both sides.
- **Line 5 — Change for exact released-code CV:** Defines five shuffled folds within the 447-row training set. It does not involve the 80-row holdout. The authors’ released cv=5 regression calls instead use KFold without this additional shuffle. Use a separate KFold(n_splits=5, shuffle=False) for that reproduction.
- **Line 6 — Optional:** Blank line separates setup from the function definition.
- **Line 7 — Change:** Defines a reusable evaluator; executing def creates the function but does not train a model yet. name labels the row; model is an estimator; four optional datasets default to the global split. store=True is never referenced in the body and can be removed without changing calls.
- **Line 8 — Keep or simplify:** Uses the global training X/y whenever the corresponding argument is None, otherwise uses supplied data. A semicolon joins two assignments. All current calls use the globals, so custom-data support is optional rather than erroneous.
- **Line 9 — Keep or simplify:** Applies the same fallback logic to the test predictors and targets. Hidden global defaults make the helper less self-contained, but they work here.
- **Line 10 — Keep unless passing fitted models explicitly:** Fits the supplied model on the chosen complete training set. For grid-searched models this duplicates GridSearchCV’s default full-training refit; an optimized evaluator can skip it when reusing best_estimator_.
- **Line 11 — Keep:** Produces predictions for training and test rows. p_tr and p_te are arrays in the same order as their respective input rows.
- **Line 12 — Keep for untuned; reuse search results for tuned:** Fits estimator clones on five training folds and returns their held-out R2 scores. It does not overwrite the fitted model. After a grid search using the same folds and best parameters, this repeats the winning candidate’s CV and is not nested CV.
- **Line 13 — Keep:** Starts the metric-row dictionary and stores the human-readable model name.
- **Line 14 — Keep:** Adds in-sample training R2. A high value alone is not evidence of generalization.
- **Line 15 — Keep + label:** Adds the average validation R2 across five folds and NumPy’s population standard deviation across those five scores. The standard deviation is descriptive fold variation, not a confidence interval or prediction interval.
- **Line 16 — Keep:** Adds R2 on the 80-row CO2 holdout. This estimates row-wise performance under the current preprocessing and split, not automatically performance for unseen materials or papers.
- **Line 17 — Keep:** Computes test RMSE by square-rooting mean squared prediction error. Its units are mmol/g; squaring penalizes larger errors more strongly.
- **Line 18 — Keep:** Computes test MAE, the mean absolute prediction error, also in mmol/g, and closes the dictionary.
- **Line 19 — Keep:** Returns four items: metric dictionary, fitted model, training predictions, and test predictions. Later callers unpack all four; deleting a returned item requires changing those callers.


##4. Chapter 3 baselines: decision tree and random forest
Chapter 3 shows that a single tree with no limits keeps splitting until each leaf is pure, which leads to overfitting. Pre-pruning (max_depth, min_samples_split, min_samples_leaf) controls this. A random forest averages many bootstrapped, decorrelated trees to reduce variance.

### Explanation: original cell 13

- **Line 1 — Change formatting:** Add the missing space after ##; this is the baseline-model section heading.
- **Line 2 — Change nuance:** Explains why limiting a tree can reduce overfitting and why forests average trees. A regression tree does not always reach perfectly pure leaves: identical predictor rows with conflicting targets cannot be separated, and other stopping conditions apply. The actual course text was not provided.


In [ ]:
rows, models, preds = [], {}, {}

r, m, ptr, pte = evaluate("Decision tree (unpruned)", DecisionTreeRegressor(random_state=SEED))
rows.append(r); models[r["model"]] = m; preds[r["model"]] = (ptr, pte)
print("Unpruned tree depth:", m.get_depth(), "| leaves:", m.get_n_leaves())

dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=SEED),
                       {"max_depth": [3, 4, 5, 6, 8, 10, None],
                        "min_samples_split": [2, 5, 10, 20],
                        "min_samples_leaf": [1, 2, 4, 8]}, cv=cv5, scoring="r2", n_jobs=-1).fit(X_train, y_train)
print("Pre-pruned tree best params:", dt_grid.best_params_)
r, m, ptr, pte = evaluate("Decision tree (pre-pruned)", DecisionTreeRegressor(random_state=SEED, **dt_grid.best_params_))
rows.append(r); models[r["model"]] = m; preds[r["model"]] = (ptr, pte)

rf_grid = GridSearchCV(RandomForestRegressor(random_state=SEED, n_jobs=-1),
                       {"n_estimators": [200, 500], "max_depth": [None, 10, 6],
                        "max_features": [1.0, 0.6, 0.33], "min_samples_leaf": [1, 2]},
                       cv=cv5, scoring="r2", n_jobs=-1).fit(X_train, y_train)
print("Random forest best params:", rf_grid.best_params_)
r, m, ptr, pte = evaluate("Random forest (tuned)", RandomForestRegressor(random_state=SEED, n_jobs=-1, **rf_grid.best_params_))
rows.append(r); models[r["model"]] = m; preds[r["model"]] = (ptr, pte)
RESULTS["dt_params"] = dt_grid.best_params_; RESULTS["rf_params"] = rf_grid.best_params_
pd.DataFrame(rows).round(3)

### Explanation: original cell 14

- **Line 1 — Keep:** Creates an empty metrics list, fitted-model dictionary, and prediction dictionary. These three containers support the main table, interpretation, and parity plots respectively.
- **Line 2 — Optional:** Blank line separates initialization from the first baseline.
- **Line 3 — Keep:** Constructs a default unpruned regression tree with fixed randomness, passes it into evaluate, and unpacks the four returned objects. Its depth is unrestricted, but default stopping rules still apply.
- **Line 4 — Keep:** Appends the metric row; stores the fitted model under its model label; stores the pair of prediction arrays under the same label. All three statements are meaningful and are separated by semicolons.
- **Line 5 — Optional recommended:** Prints the learned maximum depth and leaf count. The saved run has depth 23 and 440 leaves, consistent with a highly flexible tree.
- **Line 6 — Optional:** Blank line separates unpruned and tuned-tree experiments.
- **Line 7 — Keep:** Starts an exhaustive parameter search over DecisionTreeRegressor. GridSearchCV will clone and fit many candidate trees rather than changing one already-fitted tree.
- **Line 8 — Keep:** Tests seven depth settings, including None for unlimited depth. This grid can select an unpruned-depth candidate, so the label pre-pruned describes the search intent, not a guarantee for every dataset.
- **Line 9 — Keep:** Tests four minimum parent-node sample counts required to attempt a split. Higher values generally restrict complexity.
- **Line 10 — Keep:** Tests four minimum leaf sizes; supplies the shared five-fold validation plan, optimizes R2, allows parallel jobs, and immediately fits the search on X_train/y_train. There are 112 combinations and 560 fold fits plus the default refit.
- **Line 11 — Optional recommended:** Prints the highest-mean-CV candidate’s parameter dictionary. These settings were chosen using training-set CV, not test R2.
- **Line 12 — Optimize:** Creates a new tree with the chosen settings using ** to unpack the parameter dictionary, then trains and CV-scores it again through evaluate. It is valid but redundant; dt_grid.best_estimator_ is already fitted and dt_grid.cv_results_ already contains its fold scores.
- **Line 13 — Keep:** Stores this model’s metrics, fitted object, and predictions in the three shared containers.
- **Line 14 — Optional:** Blank line separates the decision-tree and random-forest searches.
- **Line 15 — Change parallelism recommended:** Starts the RF grid. Setting n_jobs=-1 inside the forest and again in the search nests parallel work; performance depends on the backend. A simpler policy is RF n_jobs=1 during a parallel GridSearchCV.
- **Line 16 — Keep:** Tests 200/500 trees and three depth limits. A larger forest is not the same as a deeper individual tree.
- **Line 17 — Keep + precision:** Tests feature-subset fractions and leaf sizes. With nine predictors, 1.0 considers nine, 0.6 considers five, and 0.33 considers two features at a split because scikit-learn truncates the product; 0.33 does not mean exactly three.
- **Line 18 — Keep:** Supplies folds, R2 scoring, parallel search, and starts fitting on training rows. The RF grid has 36 combinations, 180 fold fits, and one default refit.
- **Line 19 — Optional recommended:** Prints the winning RF settings. In the saved run max_features=0.33 corresponds to two candidate predictors per split.
- **Line 20 — Optimize:** Rebuilds the winning RF and redoes fitting plus five-fold scoring. Reusing rf_grid.best_estimator_ and its stored CV results avoids that repeated work.
- **Line 21 — Keep:** Stores RF metrics, fitted object, and predictions.
- **Line 22 — Keep if exporting:** Stores the selected DT and RF parameter dictionaries in RESULTS. The two assignments are joined by a semicolon.
- **Line 23 — Optional:** Builds and displays a rounded temporary table of all baseline metric rows. It does not change rows or refit models.


##5. Reproducing the paper: GBDT, XGBoost, LightGBM
5a. With the authors' final hyperparameters (taken from their published notebook).

5b. With our own grid search (5-fold CV on the training set), as the paper describes.

### Explanation: original cell 15

- **Line 1 — Change formatting:** Add a space after ##. This heading introduces the three boosted-tree reproduction models.
- **Line 2 — Keep + qualify:** Introduces models using explicit settings copied or translated from the authors’ released notebook. These are not guaranteed to recover the published table exactly with modern defaults.
- **Line 3 — Optional:** Blank line separates the two experimental tracks.
- **Line 4 — Change nuance:** Introduces your own training-only grid search. It is a new experiment, not the same hyperparameter search as the original work; notably, your LightGBM search uses a different boosting type.


In [ ]:
paper_params = {
    "GBDT": GradientBoostingRegressor(learning_rate=0.05, max_depth=4, min_samples_leaf=1, min_samples_split=2,
                                      n_estimators=300, subsample=0.5, random_state=SEED),
    "XGB": XGBRegressor(colsample_bytree=1, learning_rate=0.05, max_depth=4, n_estimators=100,
                        subsample=0.5, random_state=SEED, eval_metric="rmse"),
    "LGB": LGBMRegressor(boosting_type="dart", colsample_bytree=1, learning_rate=0.2, max_depth=4,
                         n_estimators=300, num_leaves=5, reg_lambda=0.8, subsample=1, random_state=SEED, verbose=-1),
}
for k, m in paper_params.items():
    r, m, ptr, pte = evaluate(f"{k} (authors' hyperparameters)", m)
    rows.append(r); models[r["model"]] = m; preds[r["model"]] = (ptr, pte)
pd.DataFrame(rows).round(3).tail(3)

### Explanation: original cell 16

- **Line 1 — Rename optional:** Starts a dictionary mapping short model labels to estimator objects. paper_models would be a clearer name than paper_params because the values are full models, not parameter dictionaries.
- **Line 2 — Keep:** Starts the authors-setting GBDT: learning_rate=0.05 scales each tree’s correction; max_depth=4 limits each tree; min_samples_leaf=1 and min_samples_split=2 retain default minimum sizes. Explicit defaults help document a reproduction and need not be deleted.
- **Line 3 — Keep:** Completes that GBDT with 300 boosting stages, half of the training rows sampled per stage, and fixed randomness. More trees are accumulated sequentially, unlike averaging independent RF trees.
- **Line 4 — Keep:** Starts XGBoost with all features available to each tree, learning rate 0.05, depth 4, and 100 boosting rounds. The authors’ eta argument is an alias for learning_rate; matching named parameters alone does not pin all library defaults.
- **Line 5 — Keep:** Completes XGBoost with 50% row subsampling, fixed seed, and the RMSE evaluation-metric setting. eval_metric is not the CV search scorer and, without a validation eval_set/early stopping, does not itself select a round count.
- **Line 6 — Keep + distinguish:** Starts LightGBM using DART dropout boosting, all columns, learning rate 0.2, and depth cap 4. DART is explicitly different from LightGBM’s default gbdt boosting used in your later grid.
- **Line 7 — Keep:** Completes LightGBM with 300 rounds, at most five leaves per tree, L2 penalty 0.8, full row sampling, fixed seed, and suppressed library logging. subsample=1 has no subsampling effect but documents the chosen setting; verbose=-1 controls LightGBM logs, not every Python warning.
- **Line 8 — Keep structural:** Closes the dictionary. The trailing comma on the final entry is legal; the closing brace is required.
- **Line 9 — Keep:** Iterates through model label k and estimator m for each of the three entries.
- **Line 10 — Keep:** Builds a descriptive label, calls evaluate, and replaces m with the returned fitted estimator. Reusing m is legal but a more descriptive name would improve readability.
- **Line 11 — Keep:** Adds metrics, fitted estimator, and prediction arrays for this authors-settings model to the shared containers.
- **Line 12 — Optional:** Displays only the last three metric rows, which are the three models just appended. This limits presentation, not the number of fitted models.


In [ ]:
grids = {
    "GBDT": (GradientBoostingRegressor(random_state=SEED),
             {"n_estimators": [100, 300, 500], "learning_rate": [0.02, 0.05, 0.1],
              "max_depth": [3, 4, 5], "subsample": [0.5, 0.8, 1.0]}),
    "XGB": (XGBRegressor(random_state=SEED, eval_metric="rmse", n_jobs=1),
            {"n_estimators": [100, 300, 500], "learning_rate": [0.02, 0.05, 0.1],
             "max_depth": [3, 4, 5], "subsample": [0.5, 0.8, 1.0]}),
    "LGB": (LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=1),
            {"n_estimators": [100, 300, 500], "learning_rate": [0.02, 0.05, 0.1],
             "num_leaves": [5, 15, 31], "min_child_samples": [5, 10, 20]}),
}
tuned_params = {}
for k, (est, g) in grids.items():
    gs = GridSearchCV(est, g, cv=cv5, scoring="r2", n_jobs=-1).fit(X_train, y_train)
    tuned_params[k] = gs.best_params_
    print(k, "best params:", gs.best_params_)
    r, m, ptr, pte = evaluate(f"{k} (our grid search)", est.set_params(**gs.best_params_))
    rows.append(r); models[r["model"]] = m; preds[r["model"]] = (ptr, pte)
RESULTS["tuned_params"] = tuned_params
results = pd.DataFrame(rows).set_index("model")
RESULTS["main_table"] = results.round(4).reset_index().to_dict(orient="records")
results.round(3)

### Explanation: original cell 17

- **Line 1 — Keep:** Starts a dictionary mapping each boosted-model label to a pair: base estimator and parameter grid.
- **Line 2 — Keep:** Creates the base sklearn GBDT with fixed randomness. Other parameters start at package defaults and selected ones are replaced by the search.
- **Line 3 — Keep:** Tests three boosting-round counts and three learning rates. Their effects interact; a smaller learning rate often requires more stages.
- **Line 4 — Keep:** Adds three depth limits and three row-subsampling fractions, then closes the GBDT grid and tuple. Four three-value dimensions produce 81 candidates.
- **Line 5 — Keep:** Creates XGBoost with fixed randomness, RMSE evaluation reporting, and one estimator thread so GridSearchCV can parallelize outer jobs.
- **Line 6 — Keep:** Tests three XGBoost boosting-round counts and three learning rates.
- **Line 7 — Keep:** Tests three XGBoost depths and three row-subsampling fractions; closes this 81-candidate grid and its tuple.
- **Line 8 — Change or label clearly:** Creates LightGBM without boosting_type, max_depth, or reg_lambda overrides. Defaults are gbdt, unlimited depth, and no L2 penalty, unlike the authors-setting DART/depth-4/L2-0.8 model. One thread and quiet logging support the parallel search.
- **Line 9 — Keep:** Tests three LightGBM round counts and three learning rates.
- **Line 10 — Keep:** Tests three leaf caps and three minimum child-sample counts, producing 81 combinations. This searches a different model space, not merely a better setting within the authors’ DART configuration.
- **Line 11 — Keep structural:** Closes the grids dictionary.
- **Line 12 — Keep if exporting:** Initializes a dictionary to retain each booster family’s selected parameter settings.
- **Line 13 — Keep:** Iterates over k and unpacks each dictionary value into estimator est and grid g. Parentheses describe tuple unpacking, not a function call.
- **Line 14 — Keep:** Runs exhaustive training-only five-fold R2 selection and default full-training refit. Each of these grids uses 405 fold fits plus one refit; all three use 1,215 fold fits.
- **Line 15 — Keep:** Stores the selected parameter dictionary for this model family.
- **Line 16 — Optional recommended:** Prints the model family and its chosen settings for an audit trail.
- **Line 17 — Optimize + label CV:** set_params changes the base estimator settings and returns that estimator; evaluate then fits it and repeats CV. Reuse gs.best_estimator_ and stored fold scores instead. This post-selection score is not an independent/nested assessment of the search.
- **Line 18 — Keep:** Stores this tuned model’s metrics, fitted estimator, and predictions.
- **Line 19 — Keep if exporting:** After the loop, stores all three winning parameter dictionaries in RESULTS.
- **Line 20 — Keep:** Constructs the nine-model results DataFrame and moves the model label into the row index. Later .loc lookups depend on these exact labels.
- **Line 21 — Keep if exporting:** Rounds a copy to four decimals, restores model labels as a regular column, and stores records in RESULTS for JSON. The full-precision results DataFrame remains unchanged.
- **Line 22 — Optional:** Displays the final nine-model table rounded to three decimals. Rounding can make a near-perfect score appear exactly 1.000.


##6. Comparison with Table 1 of the paper

### Explanation: original cell 18

- **Line 1 — Change formatting:** Add a space after ##. This heading introduces the reference-versus-reproduction table.


In [ ]:
paper = pd.DataFrame({"train R2": [0.98, 0.96, 0.92], "5-fold CV R2": [0.81, 0.81, 0.78],
                      "test R2": [0.84, 0.83, 0.78], "test RMSE": [0.66, 0.68, 0.77]},
                     index=["GBDT", "XGB", "LGB"])
comp = []
for k in ["GBDT", "XGB", "LGB"]:
    for src, lab in [(paper.loc[k], "Paper (Table 1)"),
                     (results.loc[f"{k} (authors' hyperparameters)"], "Ours, authors' hyperparameters"),
                     (results.loc[f"{k} (our grid search)"], "Ours, own grid search")]:
        comp.append({"model": k, "source": lab, **{c: src[c] for c in paper.columns}})
comp = pd.DataFrame(comp)
RESULTS["comparison"] = comp.round(3).to_dict(orient="records")
comp.round(2)

### Explanation: original cell 19

- **Line 1 — Verify manually:** Begins a DataFrame of hand-entered paper-reference train and CV R2 values. These numbers are not downloaded from the publication or calculated from predictions.
- **Line 2 — Verify manually:** Adds hand-entered paper test R2 and test RMSE. These values match the verified full Table 1; some published numbers differ from saved GitHub outputs.
- **Line 3 — Keep:** Assigns row labels GBDT, XGB, and LGB, then closes the reference table constructor. Array order must match these model labels.
- **Line 4 — Keep:** Creates an initially empty list that will collect three source rows per model family.
- **Line 5 — Keep:** Loops over the three model-family identifiers in a fixed display order.
- **Line 6 — Keep:** Begins an inner loop with the paper reference row and its source label; .loc[k] selects the indexed model row.
- **Line 7 — Keep:** Adds the matching current authors-hyperparameter metrics row and its descriptive label to the inner loop.
- **Line 8 — Keep:** Adds the matching own-grid-search row and label, then starts the inner loop body. Both row labels must match those created earlier.
- **Line 9 — Keep:** Appends a dictionary with model and source plus only the metrics present in paper.columns. The dictionary comprehension selects values by column name; ** merges those key/value pairs into the new dictionary. CV standard deviation and MAE are intentionally excluded here.
- **Line 10 — Keep:** Replaces the list comp with a nine-row DataFrame. Reusing a name for a new object type is valid, though naming the list comp_rows would be clearer.
- **Line 11 — Keep if exporting:** Rounds a copy of the comparison to three decimals and stores row dictionaries in RESULTS.
- **Line 12 — Optional + change precision:** Displays the comparison to two decimals. This can make 0.836 and 0.840 look identical; report three decimals when discussing reproduction errors or small improvements.


In [ ]:
best = "GBDT (authors' hyperparameters)"
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, name in zip(axes, ["GBDT (authors' hyperparameters)", "XGB (authors' hyperparameters)", "LGB (authors' hyperparameters)"]):
    ptr, pte = preds[name]
    ax.scatter(y_train, ptr, s=18, c="#E8A33D", alpha=0.6, label="Train")
    ax.scatter(y_test, pte, s=26, c="#1F4E79", alpha=0.9, label="Test")
    ax.plot([0, 8.5], [0, 8.5], "k--", lw=1)
    ax.set_xlim(0, 8.5); ax.set_ylim(0, 8.5)
    rr = results.loc[name]
    ax.set_title(f"{name.split(' ')[0]}  |  test R² = {rr['test R2']:.2f}, RMSE = {rr['test RMSE']:.2f}")
    ax.set_xlabel("Actual CO₂ uptake (mmol/g)"); ax.set_ylabel("Predicted CO₂ uptake (mmol/g)")
    ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig("figs/fig2_parity.png", dpi=200); plt.show()

### Explanation: original cell 20

- **Line 1 — Rename recommended:** Chooses the authors-setting GBDT for later interpretation, not the numerically best model in the displayed table. Rename best to interpretation_model_name everywhere it is read. Keep this assignment if Cell 23 still uses it, even if removing the parity chart.
- **Line 2 — Optional chart block:** Creates a figure and three side-by-side axes; figsize is width and height in inches. No model is fitted here.
- **Line 3 — Optional chart block:** Pairs each axis with one authors-setting model name using zip. This plot does not include your three grid-tuned boosters.
- **Line 4 — Optional chart block:** Retrieves the previously saved training/test predictions for the selected model. It does not rerun inference or training.
- **Line 5 — Optional chart block:** Plots measured training uptake on x and predicted training uptake on y; s is marker area, c is color, alpha is opacity, and label supplies legend text.
- **Line 6 — Optional chart block:** Adds the held-out test points with their own marker size, color, opacity, and legend label.
- **Line 7 — Optional chart block:** Draws the dashed black y=x identity line from 0 to 8.5; lw is line width. A point on this line is a perfect prediction. This is not a fitted regression line.
- **Line 8 — Optional chart block:** Fixes both axis ranges at 0–8.5. The current observed target maximum is 8.2; a future larger value or out-of-range prediction could be hidden. Equal limits do not by themselves force a square plotting aspect.
- **Line 9 — Optional chart block:** Retrieves this model’s metrics row to annotate the chart title.
- **Line 10 — Optional chart block:** Builds the title: name.split(" ")[0] keeps the first word, and :.2f formats test R2 and RMSE to two decimals. The superscript character is display text, not an exponent operation.
- **Line 11 — Optional chart block:** Labels x as actual and y as predicted CO2 uptake, both in mmol/g. The semicolon joins two labeling statements.
- **Line 12 — Optional chart block:** Displays the train/test legend in the upper-left corner of the current subplot.
- **Line 13 — Optional chart block:** Adjusts spacing, saves the complete figure as a 200-dpi PNG under figs, and displays it. These are three separate operations on one line. Removing savefig stops file export but not plotting.


In [ ]:
plot_models = ["Decision tree (unpruned)", "Decision tree (pre-pruned)", "Random forest (tuned)",
               "GBDT (authors' hyperparameters)", "XGB (authors' hyperparameters)", "LGB (authors' hyperparameters)"]
labels = ["DT unpruned", "DT pre-pruned", "Random forest", "GBDT", "XGBoost", "LightGBM"]
sub = results.loc[plot_models]
xx = np.arange(len(sub)); w = 0.27
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.bar(xx - w, sub["train R2"], w, label="Train R²", color="#E8A33D")
ax.bar(xx, sub["5-fold CV R2"], w, label="5-fold CV R²", color="#7F9FBF")
ax.bar(xx + w, sub["test R2"], w, label="Test R²", color="#1F4E79")
for i, v in enumerate(sub["test R2"]): ax.text(i + w, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(xx); ax.set_xticklabels(labels); ax.set_ylim(0, 1.08); ax.set_ylabel("R²")
ax.set_title("Chapter 3 tree models vs. the paper's boosted models"); ax.legend(ncol=3, loc="upper right")
plt.tight_layout(); plt.savefig("figs/fig3_model_comparison.png", dpi=200); plt.show()

### Explanation: original cell 21

- **Line 1 — Optional chart block:** Starts a list of models to include in the bar chart: the two decision trees and random forest.
- **Line 2 — Optional chart block:** Adds the three authors-setting boosters and closes the list. The grid-tuned boosters are excluded, which should be clear in the title/caption.
- **Line 3 — Optional chart block:** Provides six shorter x-axis labels in the same order as plot_models. Ordering must stay aligned if the model list changes.
- **Line 4 — Optional chart block:** Selects those six rows from the results table in the specified order.
- **Line 5 — Optional chart block:** Creates integer group positions 0–5 and sets bar width to 0.27. These are plot coordinates, not predictor values.
- **Line 6 — Optional chart block:** Creates one figure and one axes object for the grouped bar chart.
- **Line 7 — Optional chart block:** Draws training-R2 bars slightly left of each group center; w is bar width, and label/color control the legend and appearance.
- **Line 8 — Optional chart block:** Draws mean five-fold validation-R2 bars at group centers. These are not extra test-set scores.
- **Line 9 — Optional chart block:** Draws test-R2 bars slightly right of each group center.
- **Line 10 — Optional chart block:** Loops through test scores and places a two-decimal text label above each corresponding test bar. enumerate supplies bar number i and value v.
- **Line 11 — Change display robustness:** Sets tick positions and labels, fixes the y range at 0–1.08, and labels it R2. Negative R2 is possible and would be hidden below zero by these limits.
- **Line 12 — Optional chart block:** Adds the title and a three-column legend. The title compares the selected baselines with authors-setting boosters, not every model in results.
- **Line 13 — Optional chart block:** Adjusts layout, saves the 200-dpi PNG, and displays the chart. This figure is useful for explaining overfitting and is not numerically redundant with the parity plot.


##7. Model interpretation: permutation importance (MDA) and partial dependence
The paper uses permutation importance (mean decrease in accuracy). Chapter 3 warns that impurity-based importance can be misleading when features are correlated (SA, TPV, MPV), so permutation importance on the test set is the safer choice.

### Explanation: original cell 22

- **Line 1 — Change:** Add a space after ## and remove “and partial dependence” if no PDP code is added. Cell 23 computes permutation importance only.
- **Line 2 — Change scientific claim:** Held-out permutation importance is useful, but correlation can also make it misleading: a correlated predictor can substitute for the shuffled one. Do not claim it solves the SA/TPV/MPV correlation problem. The chapter wording itself was not independently checked.


In [ ]:
gbdt = models[best]
pi = permutation_importance(gbdt, X_test, y_test, n_repeats=30, random_state=SEED, scoring="r2")
imp_df = pd.DataFrame({"feature": FEATURES, "MDA (drop in R²)": pi.importances_mean, "std": pi.importances_std,
                       "impurity importance": gbdt.feature_importances_}).sort_values("MDA (drop in R²)", ascending=True)
RESULTS["importance"] = imp_df.round(4).to_dict(orient="records")
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(imp_df["feature"], imp_df["MDA (drop in R²)"], xerr=imp_df["std"], color="#1F4E79")
ax.set_xlabel("Mean decrease in test R² when feature is shuffled"); ax.set_title("GBDT permutation feature importance")
plt.tight_layout(); plt.savefig("figs/fig4_permutation_importance.png", dpi=200); plt.show()
imp_df.sort_values("MDA (drop in R²)", ascending=False).round(3)

### Explanation: original cell 23

- **Line 1 — Keep + rename dependency:** Retrieves the already fitted interpretation estimator from models using the name stored in best. Rename both this reference and Cell 20’s assignment consistently; deleting best alone causes NameError.
- **Line 2 — Keep + caveat:** Computes held-out permutation importance by shuffling each of nine features 30 times with fixed randomness and scoring the decrease in R2. It uses the existing fitted model; it does not retrain per shuffle. Test-set inspection should not become iterative feature/model selection.
- **Line 3 — Keep:** Starts a table pairing FEATURES with the mean decrease in R2 and its standard deviation over shuffles. These std values measure shuffle variability conditional on this model/data, not material-population uncertainty.
- **Line 4 — Keep if comparing:** Adds GBDT’s impurity-based importance and sorts rows by ascending permutation importance so the largest bar appears near the top of a horizontal plot. The two importance methods have different scales; impurity importances sum to one, permutation drops need not.
- **Line 5 — Keep if exporting:** Stores the ascending, rounded interpretation table as records in RESULTS. Rounding is for export, not recomputing the model.
- **Line 6 — Optional chart block:** Creates a figure and axes for the horizontal importance chart.
- **Line 7 — Optional chart block:** Draws feature importance horizontally with standard-deviation error bars. xerr does not represent a confidence interval; color is purely presentation.
- **Line 8 — Optional chart block:** Labels the metric as test-R2 decrease after shuffling and titles the chart for GBDT. A decrease of 0.709 does not mean 70.9% of adsorption is caused by that feature.
- **Line 9 — Optional chart block:** Adjusts layout, saves the chart to a 200-dpi PNG, and displays it.
- **Line 10 — Optional:** Displays a separate descending-order, three-decimal table for reading. It does not change the previously stored imp_df sort order or refit the model.


##10. Save results

### Explanation: original cell 24

- **Line 1 — Change:** Renumber this to section 8 and add a space after ##. Sections 8 and 9 do not exist in the supplied notebook; the numbering jump does not affect execution.


In [ ]:
RESULTS["versions"] = {"scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__}
with open("results.json", "w") as f: json.dump(RESULTS, f, indent=2, default=float)
results.round(3).to_csv("results_main_table.csv")
print("Saved results.json, results_main_table.csv, and figures in ./figs")

### Explanation: original cell 25

- **Line 1 — Keep recommended:** Adds three package-version strings to RESULTS. This records the current environment, but does not save Python/NumPy/pandas versions, a lockfile, a data commit/hash, split indexes, or fitted models.
- **Line 2 — Optional export block:** Opens results.json for writing; the with context closes it automatically. json.dump writes the dictionary with two-space indentation. default=float converts unsupported scalar values when possible; it is not a universal serializer. Rerunning overwrites the file.
- **Line 3 — Optional export block:** Writes the main results table, rounded to three decimals, to CSV. The default index=True includes the model names because model is the DataFrame index. It does not save model weights.
- **Line 4 — Optional:** Prints what was saved. Update this message if any export or chart block is removed; it is not a check that all expected files actually exist.
